<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/06-getting-data-out-safely.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 3.6 — show / collect / take / toPandas, safely

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("3.6-getting-data-out")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## show(): the safe window and its dials

In [ ]:
orders.show(3)
orders.show(2, truncate=False)
orders.show(1, vertical=True)
print("show returns:", orders.show(1))   # None — it prints, it isn't data

## collect(): Row objects, both access styles — safe here because 41 rows

In [ ]:
rows = orders.collect()
r = rows[0]
print(len(rows), "rows on the driver")
print(r.product, "|", r["unit_price"], "|", r.asDict()["country"])

In [ ]:
# The LEGITIMATE collect: after aggregation, feeding Python logic
cat_revenue = (
    orders.groupBy("category")
    .agg(F.sum(col("quantity") * col("unit_price")).alias("revenue"))
    .collect()                    # 5 rows — arithmetic known safe
)
thresholds = {row["category"]: row["revenue"] > 100 for row in cat_revenue}
thresholds

## take / first / count — bounded exits

In [ ]:
print(orders.take(2))
print(orders.first().order_id)
print(orders.count())
# 'which rows' take() returns is arbitrary — top-N needs orderBy first (lesson 3.3)

## toPandas(): the production idiom — reduce in Spark, decorate in pandas

In [ ]:
print("Arrow enabled:", spark.conf.get("spark.sql.execution.arrow.pyspark.enabled"))

pdf = (
    orders
    .groupBy("order_date")
    .agg(F.round(F.sum(col("quantity") * col("unit_price")), 2).alias("revenue"))
    .orderBy("order_date")
    .toPandas()                   # only the daily totals cross the wire
)
print(type(pdf), pdf.shape)
pdf.head()

In [ ]:
# ...and now the pandas ecosystem is available on the RESULT:
ax = pdf.plot(x="order_date", y="revenue", kind="line", title="Daily revenue")
ax.figure.tight_layout()

## The anti-pattern, named and shamed

```python
pdf = spark.read.parquet("s3://.../events").toPandas()   # ← cluster as file-downloader
pdf = pdf[pdf.status == "active"]                          # ← Spark could do this
```
Aggregate/filter/sample in Spark **first**; `toPandas()` is for results,
not for data. (It works on our 41 rows — that's exactly why the habit
forms in notebooks and detonates in production.)

## Exercises

1. Fix the anti-pattern above for: "histogram of line_total for IN electronics orders" — what runs in Spark, what lands in pandas?
2. `orders.toPandas().groupby("category").revenue.sum()` vs the 3.4 groupBy + small toPandas — same numbers? Which scales, and what exactly breaks the other one first?
3. Use `take(3)` in a loop to print order summaries WITHOUT `collect()` — then explain why even this take is wasteful if you also need the count (hint: two actions, lesson 2.1).
4. Sample 25% of orders (seeded), reduce to per-country averages, `toPandas()`, and bar-plot it — the full safe-EDA loop in one chain.

In [ ]:
# your exercise code here
